# 10 — Augmentacja klas rzadkich: ablacja (HerBERT)

Te same warunki co w 09, na HerBERT-base: `baseline` (gołe BCE), `reweight` (pos_weight), `oversample` (`WeightedRandomSampler`), `back-translation`. Test nietknięty.

> **Uwaga:** zapisane niżej wyjścia komórek pochodzą z runu sprzed wygenerowania
> `aug_llm_train.csv`, więc brakuje w nich warunku `5_llm`. Ponowne uruchomienie
> notatnika odtwarza pełną ablację.

In [ ]:
import warnings
from pathlib import Path
import numpy as np, pandas as pd, torch, torch.nn.functional as F
from scipy.special import expit
from sklearn.metrics import f1_score, recall_score
from datasets import Dataset
from torch.utils.data import WeightedRandomSampler
from transformers import (AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments,
                          Trainer, DataCollatorWithPadding, EarlyStoppingCallback)
from thesis_lib import optimal_thresholds as opt_thresholds
warnings.filterwarnings("ignore")

EMOTIONS=["radość","smutek","zaufanie","wstręt","strach","gniew","przeczuwanie","zdziwienie"]
RARE=["strach","zaufanie","smutek"]; RANDOM_STATE=42
torch.manual_seed(RANDOM_STATE); np.random.seed(RANDOM_STATE)
PROCESSED_DIR=Path("../data/processed"); RESULTS_DIR=Path("../data/results")
FIGURES_DIR=Path("../figures"); HF_OUT=Path("../data/transformers")
MODEL="allegro/herbert-base-cased"; device="cuda" if torch.cuda.is_available() else "cpu"

tw_train=pd.read_csv(PROCESSED_DIR/"twitteremo_train.csv"); tw_val=pd.read_csv(PROCESSED_DIR/"twitteremo_val.csv")
tw_test=pd.read_csv(PROCESSED_DIR/"twitteremo_test.csv")
for d in (tw_train,tw_val,tw_test): d["tekst"]=d["tekst"].fillna("")
y_val,y_test=tw_val[EMOTIONS].values,tw_test[EMOTIONS].values
tok=AutoTokenizer.from_pretrained(MODEL)
print("device",device)

In [ ]:
def to_ds(df):
    d=Dataset.from_dict({"text":df["tekst"].tolist(),
        "labels":df[EMOTIONS].values.astype("float32").tolist()})
    return d.map(lambda b: tok(b["text"],truncation=True,max_length=128),batched=True,remove_columns=["text"])

ds_val,ds_test=to_ds(tw_val),to_ds(tw_test)

pos=tw_train[EMOTIONS].values.sum(0); neg=len(tw_train)-pos
POS_WEIGHT=torch.tensor(np.clip(neg/np.maximum(pos,1),1.0,10.0),dtype=torch.float32)

In [3]:
class AugTrainer(Trainer):
    def __init__(self,*a,pos_weight=None,sample_weights=None,**k):
        super().__init__(*a,**k); self.pos_weight=pos_weight; self.sample_weights=sample_weights
    def compute_loss(self,model,inputs,return_outputs=False,**kw):
        labels=inputs.pop("labels"); out=model(**inputs)
        loss=F.binary_cross_entropy_with_logits(out.logits.float(),labels.float(),
              pos_weight=self.pos_weight.to(out.logits.device) if self.pos_weight is not None else None)
        return (loss,out) if return_outputs else loss
    def _get_train_sampler(self,*a,**k):
        if self.sample_weights is not None:
            return WeightedRandomSampler(self.sample_weights,len(self.sample_weights),replacement=True)
        return super()._get_train_sampler(*a,**k)

In [4]:
def run(condition, train_df, pos_weight=None, oversample=False, epochs=4):
    print(f"\n=== {condition} (n_train={len(train_df)}) ===",flush=True)
    ds_train=to_ds(train_df)
    model=AutoModelForSequenceClassification.from_pretrained(
        MODEL,num_labels=len(EMOTIONS),problem_type="multi_label_classification")
    sw=None
    if oversample:
        rare_row=(train_df[RARE].sum(1)>0).values.astype(float)
        sw=(1.0+4.0*rare_row).tolist()   # rzadkie x5 częściej losowane
    args=TrainingArguments(output_dir=str(HF_OUT/f"aug_{condition}"),
        eval_strategy="epoch",save_strategy="epoch",save_total_limit=1,
        load_best_model_at_end=True,metric_for_best_model="f1_macro",greater_is_better=True,
        per_device_train_batch_size=8,per_device_eval_batch_size=32,gradient_accumulation_steps=2,
        gradient_checkpointing=True,num_train_epochs=epochs,learning_rate=2e-5,warmup_ratio=0.1,
        weight_decay=0.01,fp16=torch.cuda.is_available(),logging_steps=100,report_to="none",seed=RANDOM_STATE)
    def cm(p): return {"f1_macro":f1_score(p.label_ids.astype(int),(expit(p.predictions)>=0.5).astype(int),average="macro",zero_division=0)}
    tr=AugTrainer(model=model,args=args,train_dataset=ds_train,eval_dataset=ds_val,
        data_collator=DataCollatorWithPadding(tok),compute_metrics=cm,pos_weight=pos_weight,
        sample_weights=sw,callbacks=[EarlyStoppingCallback(early_stopping_patience=2)])
    tr.train()
    thr=opt_thresholds(y_val,expit(tr.predict(ds_val).predictions))
    pred=(expit(tr.predict(ds_test).predictions)>=thr).astype(int)
    row={"warunek":condition,"n_train":len(train_df),
         "f1_macro":f1_score(y_test,pred,average="macro",zero_division=0),
         "f1_micro":f1_score(y_test,pred,average="micro",zero_division=0)}
    for c in RARE:
        i=EMOTIONS.index(c); row[f"recall_{c}"]=recall_score(y_test[:,i],pred[:,i],zero_division=0)
    freq=[e for e in EMOTIONS if e not in RARE]
    row["f1_frequent_avg"]=np.mean([f1_score(y_test[:,EMOTIONS.index(e)],pred[:,EMOTIONS.index(e)],zero_division=0) for e in freq])
    del tr,model; torch.cuda.empty_cache()
    print(f"  -> F1-Macro={row['f1_macro']:.3f}")
    return row

In [5]:
variants=[("1_baseline",tw_train,{}),
          ("2_reweight",tw_train,{"pos_weight":POS_WEIGHT}),
          ("3_oversample",tw_train,{"oversample":True})]
bt_path=PROCESSED_DIR/"aug_bt_train.csv"
if bt_path.exists():
    bt=pd.read_csv(bt_path); bt["tekst"]=bt["tekst"].fillna("")
    aug_bt=pd.concat([tw_train,bt[["tekst"]+EMOTIONS]],ignore_index=True)
    variants.append(("4_backtranslation",aug_bt,{}))
llm_path=PROCESSED_DIR/"aug_llm_train.csv"
if llm_path.exists():
    llm=pd.read_csv(llm_path); llm["tekst"]=llm["tekst"].fillna("")
    variants.append(("5_llm",pd.concat([tw_train,llm[["tekst"]+EMOTIONS]],ignore_index=True),{}))

rows=[run(name,df,**kw) for name,df,kw in variants]
res=pd.DataFrame(rows)
res.to_csv(RESULTS_DIR/"aug_transformer_ablation.csv",index=False)
display(res.round(3))


=== 1_baseline (n_train=28684) ===


Map:   0%|          | 0/28684 [00:00<?, ? examples/s]

Map:   7%|▋         | 2000/28684 [00:00<00:02, 12294.89 examples/s]

Map:  14%|█▍        | 4000/28684 [00:00<00:02, 12034.01 examples/s]

Map:  21%|██        | 6000/28684 [00:00<00:01, 12141.86 examples/s]

...

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.45it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

There were unexpected keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.beta', 'bert.embeddings.LayerNorm.gamma', 'bert.encoder.layer.0.attention.output.LayerNorm.beta', 'bert.encoder.layer.0.attention.output.LayerNorm.gamma', 'bert.encoder.layer.0.output.LayerNorm.beta', 'bert.encoder.layer.0.output.LayerNorm.gamma', 'bert.encoder.layer.1.attention.output.LayerNorm.beta', 'bert.encoder.layer.1.attention.output.LayerNorm.gamma', 'bert.encoder.layer.1.output.LayerNorm.beta', 'bert.encoder.layer.1.output.LayerNorm.gamma', 'bert.encoder.layer.2.attention.output.LayerNorm.beta', 'bert.encoder.layer.2.attention.output.LayerNorm.gamma', 'bert.encoder.layer.2.output.LayerNorm.beta', 'bert.encoder.layer.2.output.LayerNorm.gamma', 'bert.encoder.layer.3.attention.output.LayerNorm.beta', 'bert.encoder.layer.3.attention.output.LayerNorm.gamma', 'bert.encoder.layer.3.output.LayerNorm.beta', 'bert.encoder.layer.3.output.LayerNorm.gamma', 'bert.encoder.layer.4.attention.output.LayerNor

  -> F1-Macro=0.535


,warunek,n_train,f1_macro,f1_micro,recall_strach,recall_zaufanie,recall_smutek,f1_frequent_avg
0,1_baseline,28684,0.539,0.636,0.154,0.656,0.448,0.635
1,2_reweight,28684,0.548,0.649,0.231,0.422,0.507,0.645
2,3_oversample,28684,0.540,0.633,0.308,0.656,0.433,0.622
3,4_backtranslation,31702,0.535,0.644,0.154,0.375,0.418,0.638
